# Music Annotation Project — Notebook 35
## V8.2f Read-Only Reconstruction-Avoided Counterfactual Audit

**Runtime:** CPU. No model training or inference is performed.

This notebook compares four development-only views of candidate usefulness:

1. **Current pixel ownership truth** — PRINT / ANNOTATION / UNCERTAIN from the exact V8.2 truth table.
2. **Saved calibrated `p_print`** — unchanged source-group OOF probability from V8.2.
3. **Counterfactual reconstruction avoided** — whether copying the observed byte at a candidate is closer to the clean page than retaining the V7.8 byte.
4. **Local/group structural gain** — changes in clean-ink agreement, tolerant edge F1 and clean-skeleton recall after restoring a near-collinear group.

### Scope and restrictions

- read-only with respect to V7.5, V8.2 and Notebook 32 artefacts;
- no retraining, refitting, recalibration, threshold selection or relabelling;
- no independent pages;
- clean truth is used only for this post-hoc development audit;
- no result is a runtime feature, production rule or promotion decision;
- every counterfactual starts from the exact V7.8 page and changes only declared candidate coordinates to exact observed bytes.

The central pixel counterfactual is:

```text
absolute-error gain = |V7.8 - clean| - |observed - clean|
```

Positive values mean preservation is closer to clean at that coordinate; negative values mean V7.8 is closer. This is an output-quality diagnostic, not a replacement truth label.

Near-collinear groups are formed only for analysis. Their gap and orientation tolerances are explicit below. Group utility is a declared descriptive composite; the notebook also retains every constituent metric so the composite cannot conceal trade-offs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scipy scikit-image openpyxl

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, sys, shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, kendalltau
from skimage.morphology import skeletonize

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V82C_ROOT=ROOT/'observed_print_classifier_v8_2c_label_sensitivity'
V82D_ROOT=ROOT/'observed_print_classifier_v8_2d_ownership_review'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2f_reconstruction_avoided'
SOURCE_V82C_RUN_OVERRIDE=None
SOURCE_V82_RUN_OVERRIDE=None
SOURCE_V82D_RUN_OVERRIDE=None  # optional; ownership reviews are joined when available

# Candidate population used to construct near-collinear groups.
GROUP_MIN_P_PRINT=0.50
GROUP_REQUIRE_POSITIVE_DIRECT_GAIN=True
GROUP_INCLUDE_ALREADY_PRINT=False
COLLINEAR_MAX_GAP=4       # maximum missing-pixel gap; neighbour centre distance <= gap + 1
COLLINEAR_PERP_TOL=1      # row/column/diagonal deviation
GROUP_MIN_PIXELS=2
GROUP_PATCH_MARGIN=28
LOCAL_EVAL_DILATION=3
MAX_GROUP_VISUALS=100
TOP_K_VALUES=[50,100,250,500,1000]

# Diagnostic composite only. All component metrics are exported separately.
UTILITY_WEIGHTS={
    'mae_total_gain':1.0,
    'ink_f1_gain_per_eval_pixel':255.0,
    'edge_f1_gain_per_eval_pixel':255.0,
    'skeleton_recall_gain_per_eval_pixel':255.0,
    'worsened_pixel_penalty':5.0,
}

ADAPTIVE_BLOCK_SIZE=41; ADAPTIVE_C=15
INK_THRESHOLD=200
EDGE_LOW=70; EDGE_HIGH=170
METRIC_TOLERANCE=1
SEED=42
COPY_SOURCE_RECORD=True
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2f_reconstruction_avoided'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; VISUAL=OUT/'visual_audit'; GROUPS=OUT/'group_counterfactuals'; RECORD=OUT/'source_record'
for p in [OUT,TABLES,AUDIT,VISUAL,GROUPS,RECORD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Resolve and verify the Notebook 32 → V8.2 lineage

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}
def complete_runs(root,required):
    if not Path(root).exists(): return []
    return sorted([p for p in Path(root).iterdir() if p.is_dir() and all((p/r).exists() for r in required)],key=lambda p:p.stat().st_mtime)

req32=['experiment_identity.json','tables/all_eligible_label_sensitivity.parquet']
if SOURCE_V82C_RUN_OVERRIDE:
    V82C=Path(SOURCE_V82C_RUN_OVERRIDE)
else:
    hits=complete_runs(V82C_ROOT,req32)
    if not hits: raise RuntimeError(f'No complete Notebook 32 run below {V82C_ROOT}')
    V82C=hits[-1]
id32=json.loads((V82C/'experiment_identity.json').read_text())
if id32.get('version')!='8.2c-whole-population-label-sensitivity-audit':
    raise RuntimeError(f"Unexpected Notebook 32 identity: {id32.get('version')}")

V82=Path(SOURCE_V82_RUN_OVERRIDE) if SOURCE_V82_RUN_OVERRIDE else Path(id32['source_v82_run'])
required82=['experiment_identity.json','tables/all_eligible_oof_predictions.parquet','tables/eligible_pixel_manifest.parquet','audit/input_and_label_audit.csv','whole_pages']
missing=[r for r in required82 if not (V82/r).exists()]
if missing: raise RuntimeError(f'V8.2 source incomplete: {missing}')
id82=json.loads((V82/'experiment_identity.json').read_text())
if id82.get('version')!='8.2-development-whole-page' or id82.get('independent_data_used') is not False:
    raise RuntimeError('Notebook 35 requires the development-only V8.2 OOF source run.')

enriched=pd.read_parquet(V82C/'tables'/'all_eligible_label_sensitivity.parquet')
pred=pd.read_parquet(V82/'tables'/'all_eligible_oof_predictions.parquet')
for df in [enriched,pred]:
    df['page']=df.page.astype(str)
    for c in ['y','x','truth','fold']: df[c]=pd.to_numeric(df[c],errors='raise').astype(int)
    df['p_print']=pd.to_numeric(df.p_print,errors='raise').astype(float)
if enriched.duplicated(['page','y','x']).any() or pred.duplicated(['page','y','x']).any(): raise RuntimeError('Duplicate page/y/x rows.')
check=enriched[['page','y','x','truth','fold','p_print']].merge(pred[['page','y','x','truth','fold','p_print']],on=['page','y','x'],suffixes=('_32','_82'),validate='one_to_one')
if len(check)!=len(pred) or not np.array_equal(check.truth_32.to_numpy(),check.truth_82.to_numpy()) or not np.array_equal(check.fold_32.to_numpy(),check.fold_82.to_numpy()) or not np.allclose(check.p_print_32,check.p_print_82,rtol=0,atol=1e-12):
    raise RuntimeError('Notebook 32 is not an exact enrichment of the same V8.2 rows/probabilities.')

# Optional human ownership review.
ownership=None; V82D=None
if SOURCE_V82D_RUN_OVERRIDE:
    candidates=[Path(SOURCE_V82D_RUN_OVERRIDE)]
else:
    candidates=complete_runs(V82D_ROOT,['experiment_identity.json','tables/ownership_review.csv'])
if candidates:
    candidate=candidates[-1]; iid=json.loads((candidate/'experiment_identity.json').read_text())
    if iid.get('version')=='8.2d-distance-two-ownership-review' and Path(iid.get('source_notebook32_run',''))==V82C:
        V82D=candidate; ownership=pd.read_csv(V82D/'tables'/'ownership_review.csv').fillna('')
        ownership['page']=ownership.page.astype(str)
        for c in ['y','x']: ownership[c]=pd.to_numeric(ownership[c],errors='raise').astype(int)
        if ownership.duplicated(['page','y','x']).any(): raise RuntimeError('Duplicate ownership-review coordinates.')

print('Notebook 32:',V82C)
print('V8.2:',V82)
print('Eligible rows:',len(enriched),'Pages:',enriched.page.nunique(),'Source groups:',enriched.source_group.nunique())
print('Ownership review:',V82D if V82D else 'not joined')

# Capture immutable source hashes before any audit calculations.
SOURCE_FILES=[V82/'experiment_identity.json',V82/'tables'/'all_eligible_oof_predictions.parquet',V82/'tables'/'eligible_pixel_manifest.parquet',V82/'audit'/'input_and_label_audit.csv',V82C/'experiment_identity.json',V82C/'tables'/'all_eligible_label_sensitivity.parquet']
if V82D: SOURCE_FILES += [V82D/'experiment_identity.json',V82D/'tables'/'ownership_review.csv']
SOURCE_LOCKS={str(path):sha256_file(path) for path in SOURCE_FILES}


## 3. Resolve exact clean, observed and V7.5 cache inputs

In [ ]:
CLEAN_MAP=image_map(ROOT/'source'/'clean'); OBSERVED_MAP=image_map(ROOT/'aligned')
V75=Path(id82['v7_5_reference'])
status=pd.read_csv(V75/'tables'/'cache_status.csv'); CACHE={str(r.page):Path(r.cache) for _,r in status.iterrows()}
input_audit=pd.read_csv(V82/'audit'/'input_and_label_audit.csv'); input_audit['page']=input_audit.page.astype(str); input_audit=input_audit.set_index('page')
pages=sorted(enriched.page.unique()); resolved=[]
for page in pages:
    for name,m in [('clean',CLEAN_MAP),('observed',OBSERVED_MAP),('cache',CACHE)]:
        if page not in m: raise RuntimeError(f'{page}: {name} input missing')
    if page not in input_audit.index: raise RuntimeError(f'{page}: absent from V8.2 input audit')
    expected_clean=str(input_audit.loc[page,'clean_sha256']); expected_obs=str(input_audit.loc[page,'observed_sha256']); expected_cache=str(input_audit.loc[page,'cache_sha256'])
    if sha256_file(CLEAN_MAP[page])!=expected_clean: raise RuntimeError(f'{page}: clean hash mismatch')
    if sha256_file(OBSERVED_MAP[page])!=expected_obs: raise RuntimeError(f'{page}: observed hash mismatch')
    if sha256_file(CACHE[page])!=expected_cache: raise RuntimeError(f'{page}: V7.5 cache hash mismatch')
    resolved.append({'page':page,'clean_path':str(CLEAN_MAP[page]),'observed_path':str(OBSERVED_MAP[page]),'cache_path':str(CACHE[page]),'clean_sha256':expected_clean,'observed_sha256':expected_obs,'cache_sha256':expected_cache})
pd.DataFrame(resolved).to_csv(AUDIT/'resolved_and_hash_verified_inputs.csv',index=False)
print('Hash-verified pages:',len(resolved))

## 4. Pixel-level reconstruction-avoided counterfactual

For every eligible coordinate, this cell compares the exact V7.8 byte with the exact observed byte against clean truth. It does **not** modify the stored page.

The table retains:

- current ownership truth and optional reviewed ownership;
- `p_print` and V8.2 decision;
- absolute- and squared-error gain;
- direct clean-ink match gain;
- observed and V7.8 edge agreement near the candidate;
- V7.8 route/support context and Notebook 32 morphology evidence.

In [ ]:
TRUTH_NAME={0:'ANNOTATION',1:'PRINT',2:'UNCERTAIN'}
def printed_ink(img): return cv2.adaptiveThreshold(img,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,ADAPTIVE_BLOCK_SIZE,ADAPTIVE_C)>0
def tolerant_match_map(ref,cand,tol=1):
    k=np.ones((2*tol+1,2*tol+1),np.uint8); return cv2.dilate(ref.astype(np.uint8),k)>0,cv2.dilate(cand.astype(np.uint8),k)>0

pixel_pages=[]
needed_cache=['geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact','mask']
for page,g0 in enriched.groupby('page',sort=False):
    clean=cv2.imread(str(CLEAN_MAP[page]),0); obs=cv2.imread(str(OBSERVED_MAP[page]),0)
    with np.load(CACHE[page]) as z:
        miss=[k for k in needed_cache if k not in z.files]
        if miss: raise RuntimeError(f'{page}: cache missing {miss}')
        a={k:z[k] for k in needed_cache}
    v78=a['geometry_t80'];
    if clean is None or obs is None or not(clean.shape==obs.shape==v78.shape): raise RuntimeError(f'{page}: image shape/read failure')
    if not np.array_equal(v78[a['mask']==0],obs[a['mask']==0]): raise RuntimeError(f'{page}: V7.8 outside-mask mismatch')
    g=g0.copy().reset_index(drop=True); ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    cv=clean[ys,xs].astype(np.int16); ov=obs[ys,xs].astype(np.int16); bv=v78[ys,xs].astype(np.int16)
    clean_ink=printed_ink(clean); obs_ink=printed_ink(obs); base_ink=printed_ink(v78)
    clean_edge=cv2.Canny(clean,EDGE_LOW,EDGE_HIGH)>0; obs_edge=cv2.Canny(obs,EDGE_LOW,EDGE_HIGH)>0; base_edge=cv2.Canny(v78,EDGE_LOW,EDGE_HIGH)>0
    ref_ed,obs_ed_d=tolerant_match_map(clean_edge,obs_edge,METRIC_TOLERANCE); _,base_ed_d=tolerant_match_map(clean_edge,base_edge,METRIC_TOLERANCE)
    g['truth_name']=g.truth.map(TRUTH_NAME); g['clean_value']=cv; g['observed_value']=ov; g['v78_value']=bv
    g['baseline_abs_error']=np.abs(bv-cv); g['preserve_abs_error']=np.abs(ov-cv); g['reconstruction_avoided_mae_gain']=g.baseline_abs_error-g.preserve_abs_error
    g['baseline_sq_error']=(bv-cv).astype(np.int32)**2; g['preserve_sq_error']=(ov-cv).astype(np.int32)**2; g['reconstruction_avoided_mse_gain']=g.baseline_sq_error-g.preserve_sq_error
    g['counterfactual_improves']=g.reconstruction_avoided_mae_gain>0; g['counterfactual_worsens']=g.reconstruction_avoided_mae_gain<0; g['counterfactual_ties']=g.reconstruction_avoided_mae_gain==0
    g['clean_ink_at_centre']=clean_ink[ys,xs]; g['observed_ink_at_centre']=obs_ink[ys,xs]; g['v78_ink_at_centre']=base_ink[ys,xs]
    g['direct_ink_match_gain']=(obs_ink[ys,xs]==clean_ink[ys,xs]).astype(np.int8)-(base_ink[ys,xs]==clean_ink[ys,xs]).astype(np.int8)
    g['observed_edge_matches_clean_nearby']=(obs_edge[ys,xs]&ref_ed[ys,xs]) | (clean_edge[ys,xs]&obs_ed_d[ys,xs])
    g['v78_edge_matches_clean_nearby']=(base_edge[ys,xs]&ref_ed[ys,xs]) | (clean_edge[ys,xs]&base_ed_d[ys,xs])
    g['direct_edge_match_gain']=g.observed_edge_matches_clean_nearby.astype(np.int8)-g.v78_edge_matches_clean_nearby.astype(np.int8)
    # Exact route/support values from the verified cache.
    route=np.full(len(g),'none',object); route[a['g80_preserve'][ys,xs]>0]='preserve'; route[a['g80_remove'][ys,xs]>0]='remove'; route[a['g80_ambiguous'][ys,xs]>0]='ambiguous'; g['route_replayed']=route
    g['horizontal_support_replayed']=a['g80_horizontal'][ys,xs]>0; g['vertical_support_replayed']=a['g80_vertical'][ys,xs]>0; g['compact_support_replayed']=a['g80_compact'][ys,xs]>0
    pixel_pages.append(g)
pixels=pd.concat(pixel_pages,ignore_index=True)
if ownership is not None:
    cols=[c for c in ['page','y','x','ownership_class','structure_type','ownership_note'] if c in ownership.columns]
    pixels=pixels.merge(ownership[cols],on=['page','y','x'],how='left',validate='one_to_one')
    for c in set(cols)-{'page','y','x'}: pixels[c]=pixels[c].fillna('')
else: pixels['ownership_class']=''; pixels['structure_type']=''; pixels['ownership_note']=''
pixels['already_v82_print']=pixels.decision.eq('PRINT')
pixels.to_parquet(TABLES/'pixel_counterfactual_audit.parquet',index=False)
pixels.to_csv(TABLES/'pixel_counterfactual_audit.csv',index=False)
print(pixels[['counterfactual_improves','counterfactual_worsens','counterfactual_ties']].sum())

## 5. Pixel-level summaries and ranking comparison

In [ ]:
summary_by_truth=(pixels.groupby(['truth','truth_name'],as_index=False).agg(
    pixels=('page','size'),pages=('page','nunique'),mean_p_print=('p_print','mean'),median_p_print=('p_print','median'),
    positive_gain_pixels=('counterfactual_improves','sum'),worsened_pixels=('counterfactual_worsens','sum'),
    total_mae_gain=('reconstruction_avoided_mae_gain','sum'),mean_mae_gain=('reconstruction_avoided_mae_gain','mean'),
    direct_ink_match_gain=('direct_ink_match_gain','sum'),direct_edge_match_gain=('direct_edge_match_gain','sum')
))
summary_by_truth['positive_gain_rate']=summary_by_truth.positive_gain_pixels/summary_by_truth.pixels
summary_by_truth.to_csv(TABLES/'pixel_counterfactual_summary_by_truth.csv',index=False)

summary_by_decision=(pixels.groupby('decision',as_index=False).agg(pixels=('page','size'),pages=('page','nunique'),positive_gain_pixels=('counterfactual_improves','sum'),worsened_pixels=('counterfactual_worsens','sum'),total_mae_gain=('reconstruction_avoided_mae_gain','sum'),mean_p_print=('p_print','mean')))
summary_by_decision.to_csv(TABLES/'pixel_counterfactual_summary_by_decision.csv',index=False)

# Ownership score is descriptive: ANNOTATION=0, UNCERTAIN/mixed/registration ambiguity=.5, PRINT=1.
ownership_score=pixels.truth.map({0:0.0,1:1.0,2:0.5}).astype(float)
review_map={'Annotation':0.0,'Printed structure':1.0,'Registration ambiguity':0.5,'Mixed / indeterminate ownership':0.5}
review_score=pixels.ownership_class.map(review_map)
pixels['ownership_rank_score']=ownership_score
pixels['reviewed_ownership_rank_score']=review_score

rank_rows=[]
metrics={'current_ownership_score':ownership_score,'p_print':pixels.p_print,'reconstruction_avoided_mae_gain':pixels.reconstruction_avoided_mae_gain,'direct_ink_match_gain':pixels.direct_ink_match_gain,'direct_edge_match_gain':pixels.direct_edge_match_gain}
for a,sa in metrics.items():
    for b,sb in metrics.items():
        mask=pd.notna(sa)&pd.notna(sb)
        rho,pv=spearmanr(np.asarray(sa)[mask],np.asarray(sb)[mask]); tau,tp=kendalltau(np.asarray(sa)[mask],np.asarray(sb)[mask])
        rank_rows.append({'metric_a':a,'metric_b':b,'rows':int(mask.sum()),'spearman_rho':rho,'spearman_p':pv,'kendall_tau':tau,'kendall_p':tp})
pd.DataFrame(rank_rows).to_csv(TABLES/'pixel_rank_correlations.csv',index=False)

# Top-k overlap between p_print and direct reconstruction benefit.
overlap=[]
valid=pixels[~pixels.already_v82_print].copy()
for k in TOP_K_VALUES:
    kk=min(k,len(valid)); a=set(map(tuple,valid.nlargest(kk,'p_print')[['page','y','x']].to_numpy())); b=set(map(tuple,valid.nlargest(kk,'reconstruction_avoided_mae_gain')[['page','y','x']].to_numpy()))
    overlap.append({'k':kk,'intersection':len(a&b),'union':len(a|b),'overlap_fraction_of_k':len(a&b)/max(kk,1),'jaccard':len(a&b)/max(len(a|b),1)})
pd.DataFrame(overlap).to_csv(TABLES/'top_k_p_print_vs_reconstruction_gain_overlap.csv',index=False)

if ownership is not None:
    reviewed=pixels[pixels.ownership_class.astype(str).str.len()>0].copy()
    reviewed.groupby('ownership_class',as_index=False).agg(pixels=('page','size'),mean_p_print=('p_print','mean'),median_p_print=('p_print','median'),positive_gain_rate=('counterfactual_improves','mean'),mean_mae_gain=('reconstruction_avoided_mae_gain','mean'),total_mae_gain=('reconstruction_avoided_mae_gain','sum')).to_csv(TABLES/'reviewed_ownership_vs_counterfactual_gain.csv',index=False)
print(summary_by_truth.to_string(index=False))

## 6. Near-collinear candidate groups

The graph links candidates on the same page when they are separated by at most the configured gap and are approximately horizontal, vertical or diagonal. Connected graph components are **proximity groups**, not conventional image connected components and not musical-symbol labels.

In [ ]:
def aligned_neighbour(dy,dx,max_gap=COLLINEAR_MAX_GAP,perp=COLLINEAR_PERP_TOL):
    if dy==0 and dx==0:return False
    reach=max_gap+1
    horizontal=abs(dy)<=perp and 0<abs(dx)<=reach
    vertical=abs(dx)<=perp and 0<abs(dy)<=reach
    diagonal=max(abs(dx),abs(dy))<=reach and abs(abs(dx)-abs(dy))<=perp
    return horizontal or vertical or diagonal

def group_page(g):
    pts={(int(r.y),int(r.x)):i for i,r in g.iterrows()}; unseen=set(pts); groups=[]
    offsets=[(dy,dx) for dy in range(-(COLLINEAR_MAX_GAP+1),COLLINEAR_MAX_GAP+2) for dx in range(-(COLLINEAR_MAX_GAP+1),COLLINEAR_MAX_GAP+2) if aligned_neighbour(dy,dx)]
    while unseen:
        seed=unseen.pop(); comp={seed}; stack=[seed]
        while stack:
            y,x=stack.pop()
            for dy,dx in offsets:
                p=(y+dy,x+dx)
                if p in unseen: unseen.remove(p); comp.add(p); stack.append(p)
        if len(comp)>=GROUP_MIN_PIXELS: groups.append(comp)
    return groups

pool=pixels.copy()
pool=pool[pool.p_print>=GROUP_MIN_P_PRINT]
if GROUP_REQUIRE_POSITIVE_DIRECT_GAIN: pool=pool[pool.counterfactual_improves]
if not GROUP_INCLUDE_ALREADY_PRINT: pool=pool[~pool.already_v82_print]
raw_groups=[]
for page,g in pool.groupby('page'):
    for j,coords in enumerate(group_page(g),1):
        raw_groups.append({'page':page,'local_group_number':j,'coords':coords})
print('Grouping pool:',len(pool),'Near-collinear groups:',len(raw_groups))

## 7. Group counterfactuals and local structural metrics

In [ ]:
def crop_bounds(shape,coords,margin):
    h,w=shape; ys=[p[0] for p in coords]; xs=[p[1] for p in coords]
    return max(0,min(ys)-margin),min(h,max(ys)+margin+1),max(0,min(xs)-margin),min(w,max(xs)+margin+1)
def binary_f1(ref,pred,region,tol=1):
    ref=ref&region; pred=pred&region
    if ref.sum()==0:return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); rd=cv2.dilate(ref.astype(np.uint8),k)>0; pd=cv2.dilate(pred.astype(np.uint8),k)>0
    precision=float(np.sum(pred&rd)/max(pred.sum(),1)); recall=float(np.sum(ref&pd)/max(ref.sum(),1)); return precision,recall,2*precision*recall/max(precision+recall,1e-12)
def skeleton_recall(clean_skel,pred_ink,region,tol=1):
    ref=clean_skel&region
    if ref.sum()==0:return np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8); pd=cv2.dilate(pred_ink.astype(np.uint8),k)>0
    return float(np.sum(ref&pd)/ref.sum())
def pca_orientation(coords):
    a=np.array([(x,y) for y,x in coords],float)
    if len(a)<2:return np.nan,0.0
    cov=np.cov(a,rowvar=False); vals,vecs=np.linalg.eigh(cov); order=np.argsort(vals)[::-1]; vals=vals[order]; v=vecs[:,order[0]]; angle=float(np.degrees(np.arctan2(v[1],v[0]))); linearity=float(1-vals[1]/max(vals[0],1e-12))
    return angle,linearity

group_rows=[]
for item in raw_groups:
    page=item['page']; coords=item['coords']; page_pixels=pixels[pixels.page==page].set_index(['y','x'],drop=False); q=page_pixels.loc[sorted(coords)].copy().reset_index(drop=True)
    clean=cv2.imread(str(CLEAN_MAP[page]),0); obs=cv2.imread(str(OBSERVED_MAP[page]),0)
    with np.load(CACHE[page]) as z: v78=z['geometry_t80']
    out=v78.copy(); yy=np.array([p[0] for p in coords],int); xx=np.array([p[1] for p in coords],int); out[yy,xx]=obs[yy,xx]
    changed=out!=v78
    if np.any(changed & ~np.isin(np.arange(changed.size).reshape(changed.shape),np.ravel_multi_index((yy,xx),changed.shape))): raise RuntimeError(f'{page}: unexpected group change')
    y0,y1,x0,x1=crop_bounds(clean.shape,coords,GROUP_PATCH_MARGIN); c=clean[y0:y1,x0:x1]; b=v78[y0:y1,x0:x1]; f=out[y0:y1,x0:x1]
    rel=np.zeros(c.shape,np.uint8); rel[yy-y0,xx-x0]=1; region=cv2.dilate(rel,np.ones((2*LOCAL_EVAL_DILATION+1,2*LOCAL_EVAL_DILATION+1),np.uint8))>0
    ci=printed_ink(c); bi=printed_ink(b); fi=printed_ink(f); ce=cv2.Canny(c,EDGE_LOW,EDGE_HIGH)>0; be=cv2.Canny(b,EDGE_LOW,EDGE_HIGH)>0; fe=cv2.Canny(f,EDGE_LOW,EDGE_HIGH)>0; cs=skeletonize(ci)
    bip,bir,bif=binary_f1(ci,bi,region,METRIC_TOLERANCE); fip,fir,fif=binary_f1(ci,fi,region,METRIC_TOLERANCE)
    bep,ber,bef=binary_f1(ce,be,region,METRIC_TOLERANCE); fep,fer,fef=binary_f1(ce,fe,region,METRIC_TOLERANCE)
    bsr=skeleton_recall(cs,bi,region,METRIC_TOLERANCE); fsr=skeleton_recall(cs,fi,region,METRIC_TOLERANCE)
    before=np.abs(v78[yy,xx].astype(int)-clean[yy,xx].astype(int)); after=np.abs(obs[yy,xx].astype(int)-clean[yy,xx].astype(int)); gains=before-after
    angle,linearity=pca_orientation(coords); eval_pixels=int(region.sum()); worsened=int((gains<0).sum())
    ink_gain=float(fif-bif) if np.isfinite(fif) and np.isfinite(bif) else 0.0; edge_gain=float(fef-bef) if np.isfinite(fef) and np.isfinite(bef) else 0.0; skel_gain=float(fsr-bsr) if np.isfinite(fsr) and np.isfinite(bsr) else 0.0
    total_gain=float(gains.sum())
    utility=(UTILITY_WEIGHTS['mae_total_gain']*total_gain + UTILITY_WEIGHTS['ink_f1_gain_per_eval_pixel']*eval_pixels*ink_gain + UTILITY_WEIGHTS['edge_f1_gain_per_eval_pixel']*eval_pixels*edge_gain + UTILITY_WEIGHTS['skeleton_recall_gain_per_eval_pixel']*eval_pixels*skel_gain - UTILITY_WEIGHTS['worsened_pixel_penalty']*float(np.abs(gains[gains<0]).sum()))
    gid=hashlib.sha256((page+'|'+';'.join(f'{y},{x}' for y,x in sorted(coords))).encode()).hexdigest()[:16]
    group_rows.append({'group_id':gid,'page':page,'source_group':q.source_group.iloc[0],'pixels':len(coords),'changed_bytes':int(changed.sum()),'xmin':int(xx.min()),'xmax':int(xx.max()),'ymin':int(yy.min()),'ymax':int(yy.max()),'orientation_deg':angle,'linearity':linearity,'mean_p_print':float(q.p_print.mean()),'max_p_print':float(q.p_print.max()),'truth_print':int((q.truth==1).sum()),'truth_annotation':int((q.truth==0).sum()),'truth_uncertain':int((q.truth==2).sum()),'mae_total_gain':total_gain,'mae_mean_gain':float(gains.mean()),'improved_pixels':int((gains>0).sum()),'worsened_pixels':worsened,'tied_pixels':int((gains==0).sum()),'ink_f1_baseline':bif,'ink_f1_forced':fif,'ink_f1_gain':ink_gain,'edge_f1_baseline':bef,'edge_f1_forced':fef,'edge_f1_gain':edge_gain,'skeleton_recall_baseline':bsr,'skeleton_recall_forced':fsr,'skeleton_recall_gain':skel_gain,'evaluation_pixels':eval_pixels,'group_utility':utility,'coordinates':'|'.join(f'{y}:{x}' for y,x in sorted(coords))})
groups=pd.DataFrame(group_rows)
if groups.empty: raise RuntimeError('No groups met the declared construction rule.')
groups['utility_rank']=groups.group_utility.rank(method='min',ascending=False).astype(int); groups['p_print_rank']=groups.mean_p_print.rank(method='min',ascending=False).astype(int); groups['mae_gain_rank']=groups.mae_total_gain.rank(method='min',ascending=False).astype(int)
groups.to_csv(TABLES/'near_collinear_group_counterfactuals.csv',index=False); groups.to_parquet(TABLES/'near_collinear_group_counterfactuals.parquet',index=False)
print(groups.sort_values('group_utility',ascending=False).head(20).to_string(index=False))

## 8. Group ranking comparison, page/source summaries and dominance flags

In [ ]:
group_corr=[]
for a,b in [('mean_p_print','group_utility'),('mean_p_print','mae_total_gain'),('mean_p_print','ink_f1_gain'),('mean_p_print','edge_f1_gain'),('mean_p_print','skeleton_recall_gain'),('truth_print','group_utility')]:
    mask=groups[[a,b]].notna().all(axis=1); rho,pv=spearmanr(groups.loc[mask,a],groups.loc[mask,b]); tau,tp=kendalltau(groups.loc[mask,a],groups.loc[mask,b]); group_corr.append({'metric_a':a,'metric_b':b,'groups':int(mask.sum()),'spearman_rho':rho,'spearman_p':pv,'kendall_tau':tau,'kendall_p':tp})
pd.DataFrame(group_corr).to_csv(TABLES/'group_rank_correlations.csv',index=False)

groups['structural_dominance_positive']=(groups.mae_total_gain>0)&(groups.ink_f1_gain>=0)&(groups.edge_f1_gain>=0)&(groups.skeleton_recall_gain>=0)&((groups.ink_f1_gain>0)|(groups.edge_f1_gain>0)|(groups.skeleton_recall_gain>0))
groups['any_structural_regression']=(groups.ink_f1_gain<0)|(groups.edge_f1_gain<0)|(groups.skeleton_recall_gain<0)
groups.to_csv(TABLES/'near_collinear_group_counterfactuals.csv',index=False)

page_summary=groups.groupby(['page','source_group'],as_index=False).agg(groups=('group_id','size'),pixels=('pixels','sum'),total_mae_gain=('mae_total_gain','sum'),positive_dominance_groups=('structural_dominance_positive','sum'),regression_groups=('any_structural_regression','sum'),total_group_utility=('group_utility','sum'))
page_summary.to_csv(TABLES/'group_results_by_page.csv',index=False)
source_summary=page_summary.groupby('source_group',as_index=False).agg(pages=('page','nunique'),groups=('groups','sum'),pixels=('pixels','sum'),total_mae_gain=('total_mae_gain','sum'),positive_dominance_groups=('positive_dominance_groups','sum'),regression_groups=('regression_groups','sum'),total_group_utility=('total_group_utility','sum'))
source_summary.to_csv(TABLES/'group_results_by_source_group.csv',index=False)

# Top-k group overlap.
group_overlap=[]
for k in TOP_K_VALUES:
    kk=min(k,len(groups)); a=set(groups.nlargest(kk,'mean_p_print').group_id); b=set(groups.nlargest(kk,'group_utility').group_id); group_overlap.append({'k':kk,'intersection':len(a&b),'overlap_fraction_of_k':len(a&b)/max(kk,1),'jaccard':len(a&b)/max(len(a|b),1)})
pd.DataFrame(group_overlap).to_csv(TABLES/'top_k_group_p_print_vs_utility_overlap.csv',index=False)

## 9. Visual audit of the highest-utility groups

In [ ]:
PURPLE=(255,0,255)
def hollow_box(im,x0,y0,x1,y1,pad=5,thickness=2):
    out=cv2.cvtColor(im,cv2.COLOR_GRAY2BGR) if im.ndim==2 else im.copy(); h,w=out.shape[:2]
    cv2.rectangle(out,(max(0,x0-pad),max(0,y0-pad)),(min(w-1,x1+pad),min(h-1,y1+pad)),PURPLE,thickness); return out
def crop_pad(im,y0,y1,x0,x1,margin=28,pad=255):
    h,w=im.shape[:2]; a=max(0,y0-margin); b=min(h,y1+margin+1); c=max(0,x0-margin); d=min(w,x1+margin+1); return im[a:b,c:d],a,c

def render_group(row):
    page=row.page; clean=cv2.imread(str(CLEAN_MAP[page]),0); obs=cv2.imread(str(OBSERVED_MAP[page]),0)
    with np.load(CACHE[page]) as z:v78=z['geometry_t80']
    coords=[tuple(map(int,p.split(':'))) for p in row.coordinates.split('|')]; yy=np.array([p[0] for p in coords]); xx=np.array([p[1] for p in coords]); forced=v78.copy(); forced[yy,xx]=obs[yy,xx]
    names=[('V7.8 baseline',v78),('Forced group',forced),('Clean diagnostic',clean),('Difference',np.where(forced!=v78,255,0).astype(np.uint8))]
    panels=[]
    for title,im in names:
        crop,oy,ox=crop_pad(im,row.ymin,row.ymax,row.xmin,row.xmax,40); panel=hollow_box(crop,row.xmin-ox,row.ymin-oy,row.xmax-ox,row.ymax-oy,4,2); panel=cv2.resize(panel,(panel.shape[1]*3,panel.shape[0]*3),interpolation=cv2.INTER_NEAREST); panels.append((title,panel))
    h=max(p.shape[0] for _,p in panels); w=max(p.shape[1] for _,p in panels); head=72; canvas=np.full((h+head,4*w,3),255,np.uint8)
    for i,(title,p) in enumerate(panels): canvas[head:head+p.shape[0],i*w:i*w+p.shape[1]]=p; cv2.putText(canvas,title,(i*w+8,24),0,.55,(20,20,20),1,cv2.LINE_AA)
    cv2.putText(canvas,f'{page} {row.group_id} pixels={row.pixels} utility={row.group_utility:.2f} p_print={row.mean_p_print:.4f}',(8,52),0,.48,(20,20,20),1,cv2.LINE_AA)
    return canvas

vis_rows=[]
for _,row in groups.sort_values(['group_utility','mae_total_gain'],ascending=False).head(MAX_GROUP_VISUALS).iterrows():
    d=VISUAL/'top_groups'; d.mkdir(parents=True,exist_ok=True); path=d/f'{int(row.utility_rank):04d}_{row.page}_{row.group_id}.png'; cv2.imwrite(str(path),render_group(row)); vis_rows.append({'group_id':row.group_id,'utility_rank':int(row.utility_rank),'page':row.page,'file':str(path.relative_to(OUT))})
pd.DataFrame(vis_rows).to_csv(AUDIT/'group_visual_inventory.csv',index=False)

# Contact sheets.
files=[OUT/r.file for _,r in pd.DataFrame(vis_rows).iterrows()]
for start in range(0,len(files),10):
    ims=[cv2.imread(str(p)) for p in files[start:start+10]]; width=max(i.shape[1] for i in ims); height=sum(i.shape[0] for i in ims); sheet=np.full((height,width,3),255,np.uint8); y=0
    for im in ims: sheet[y:y+im.shape[0],:im.shape[1]]=im; y+=im.shape[0]
    cv2.imwrite(str(VISUAL/f'top_group_contact_sheet_{start//10+1:02d}.png'),sheet)
print('Visual audit:',VISUAL)

## 10. Charts

In [ ]:
fig,ax=plt.subplots(figsize=(8,6)); ax.scatter(pixels.p_print,pixels.reconstruction_avoided_mae_gain,s=4,alpha=.18); ax.axhline(0,color='black',linewidth=1); ax.set_xlabel('Saved calibrated p_print'); ax.set_ylabel('Direct absolute-error gain'); ax.set_title('p_print versus reconstruction-avoided gain'); plt.tight_layout(); plt.savefig(VISUAL/'pixel_p_print_vs_reconstruction_gain.png',dpi=160); plt.close(fig)
fig,ax=plt.subplots(figsize=(8,6)); ax.scatter(groups.mean_p_print,groups.group_utility,s=np.clip(groups.pixels*10,10,180),alpha=.45); ax.axhline(0,color='black',linewidth=1); ax.set_xlabel('Mean saved p_print'); ax.set_ylabel('Declared diagnostic group utility'); ax.set_title('Near-collinear groups: probability versus output utility'); plt.tight_layout(); plt.savefig(VISUAL/'group_p_print_vs_utility.png',dpi=160); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,6)); q=summary_by_truth.set_index('truth_name')[['positive_gain_rate']]; q.plot(kind='bar',ax=ax,legend=False); ax.set_ylabel('Fraction with positive direct gain'); ax.set_title('Reconstruction-avoided benefit by current ownership truth'); plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(VISUAL/'positive_gain_rate_by_truth.png',dpi=160); plt.close(fig)

## 11. Read-only integrity and source record

In [ ]:
# Re-hash source files after analysis; any mutation invalidates the run.
locks=[]
for path_text,before in SOURCE_LOCKS.items():
    src=Path(path_text); after=sha256_file(src); locks.append({'path':path_text,'sha256_before':before,'sha256_after':after,'unchanged':before==after})
lock_df=pd.DataFrame(locks); lock_df.to_csv(AUDIT/'read_only_source_hash_check.csv',index=False)
if not lock_df.unchanged.all(): raise RuntimeError('A source artefact changed during Notebook 35.')
if COPY_SOURCE_RECORD:
    for path_text in SOURCE_LOCKS:
        src=Path(path_text); dest=RECORD/(src.name if src.parent.name not in ['tables','audit'] else src.parent.name+'__'+src.name); shutil.copy2(src,dest)
        if sha256_file(src)!=sha256_file(dest): raise RuntimeError(f'Copied source mismatch: {src}')
print('Read-only source hash gate passed:',len(lock_df),'files')


## 12. Identity and executive summary

In [ ]:
rank_corr=pd.read_csv(TABLES/'pixel_rank_correlations.csv'); pr=rank_corr[(rank_corr.metric_a=='p_print')&(rank_corr.metric_b=='reconstruction_avoided_mae_gain')].iloc[0]
gcorr=pd.read_csv(TABLES/'group_rank_correlations.csv'); gr=gcorr[(gcorr.metric_a=='mean_p_print')&(gcorr.metric_b=='group_utility')].iloc[0]
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.2f-reconstruction-avoided-counterfactual-audit','development_only':True,'read_only_source':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'thresholds_changed':False,'truth_labels_changed':False,'independent_data_used':False,'source_v82c_run':str(V82C),'source_v82_run':str(V82),'source_v82d_run':str(V82D) if V82D else None,'source_v82_identity_sha256':sha256_file(V82/'experiment_identity.json'),'source_v82c_identity_sha256':sha256_file(V82C/'experiment_identity.json'),'candidate_population':{'group_min_p_print':GROUP_MIN_P_PRINT,'require_positive_direct_gain':GROUP_REQUIRE_POSITIVE_DIRECT_GAIN,'include_already_print':GROUP_INCLUDE_ALREADY_PRINT},'grouping':{'max_missing_gap':COLLINEAR_MAX_GAP,'perpendicular_tolerance':COLLINEAR_PERP_TOL,'minimum_pixels':GROUP_MIN_PIXELS},'utility_weights':UTILITY_WEIGHTS,'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

lines=['V8.2f RECONSTRUCTION-AVOIDED COUNTERFACTUAL AUDIT','',f'V8.2 source: {V82}',f'Notebook 32 source: {V82C}',f'Ownership review: {V82D if V82D else "not joined"}','','Retrained: False','Refitted: False','Recalibrated: False','Truth labels changed: False','Independent data used: False','',f'Eligible pixels audited: {len(pixels)}',f'Pixels with positive direct absolute-error gain: {int(pixels.counterfactual_improves.sum())}',f'Pixels worsened by preservation: {int(pixels.counterfactual_worsens.sum())}',f'Near-collinear groups: {len(groups)}',f'Structural-dominance-positive groups: {int(groups.structural_dominance_positive.sum())}',f'Groups with any local structural regression: {int(groups.any_structural_regression.sum())}','','RANK COMPARISON',f'Pixel p_print vs direct gain Spearman rho: {pr.spearman_rho:.6f}',f'Group mean p_print vs group utility Spearman rho: {gr.spearman_rho:.6f}','','Interpretation: this post-hoc development audit compares ranking objectives. It does not define a new training target or runtime policy.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines)); print('\n'.join(lines))

## Outputs to bring back

Start with:

- `executive_summary.txt`
- `experiment_identity.json`
- `tables/pixel_counterfactual_summary_by_truth.csv`
- `tables/pixel_rank_correlations.csv`
- `tables/top_k_p_print_vs_reconstruction_gain_overlap.csv`
- `tables/near_collinear_group_counterfactuals.csv`
- `tables/group_rank_correlations.csv`
- `tables/top_k_group_p_print_vs_utility_overlap.csv`
- `tables/group_results_by_page.csv`
- `visual_audit/group_p_print_vs_utility.png`
- representative `visual_audit/top_group_contact_sheet_*.png` files.